# MURA Classification: EfficientNet-B3 + CoAtNet Head + FixMatch
**Diploma Thesis - Bone Fracture Detection**

- Shared Encoder: EfficientNet-B3 (384-d features)
- Classification Head: CoAtNet-style (Conv + Self-Attention)
- Semi-supervised: FixMatch
- Dataset: MURA (Stanford Musculoskeletal Radiographs)

## 1. Setup & Installation

In [1]:
!pip install -q timm torchmetrics

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import timm
from torchmetrics import AUROC, Accuracy, F1Score, Precision, Recall
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 95.6 kB/s eta 0:00:00
Device: cuda
GPU: Tesla T4


In [5]:
# 0. Kaggle API тохируулах
from google.colab import files
import os

uploaded = files.upload()  # kaggle.json upload
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print(' Kaggle API бэлэн')

Saving kaggle.json to kaggle.json
 Kaggle API бэлэн


## 2. MURA Dataset

In [6]:
# 2. MURA татах (~11GB — 10 минут)
!kaggle datasets download -d cjinny/mura-v11 --unzip -p /content/MURA
MURA_ROOT = '/content/MURA/MURA-v1.1'
print('MURA:', os.listdir(MURA_ROOT))

Dataset URL: https://www.kaggle.com/datasets/cjinny/mura-v11
License(s): unknown
100% 3.14G/3.14G [02:17<00:00, 24.6MB/s]

MURA: ['train_image_paths.csv', 'valid', 'valid_image_paths.csv', 'train_labeled_studies.csv', 'valid_labeled_studies.csv', 'train']


In [19]:
# MURA датасет DataFrame — өмнөх notebook-оос үүссэн байх ёстой
# Хэрэв байхгүй бол доорх кодыг ажиллуул

MURA_ROOT = '/content/MURA/MURA-v1.1'

def load_mura_df(split='train'):
    data = []
    split_path = f'{MURA_ROOT}/{split}'
    for body_part in sorted(os.listdir(split_path)):
        bp_path = f'{split_path}/{body_part}'
        if not os.path.isdir(bp_path): continue
        for patient in sorted(os.listdir(bp_path)):
            pt_path = f'{bp_path}/{patient}'
            if not os.path.isdir(pt_path): continue
            for study in sorted(os.listdir(pt_path)):
                label   = 1 if 'positive' in study else 0
                st_path = f'{pt_path}/{study}'
                if not os.path.isdir(st_path): continue
                for img_file in sorted(os.listdir(st_path)):
                    # Filter out macOS resource fork files (e.g., ._image.png)
                    if img_file.endswith('.png') and not img_file.startswith('._'):
                        data.append({
                            'path':  f'{st_path}/{img_file}',
                            'label': label
                        })
    return pd.DataFrame(data)

train_mura = load_mura_df('train')
valid_mura = load_mura_df('valid')
all_mura   = pd.concat([train_mura, valid_mura]).reset_index(drop=True)

print(f'MURA нийт зураг: {len(all_mura)}')
print(f'  Хугарал (1): {all_mura["label"].sum()}')
print(f'  Хэвийн  (0): {(all_mura["label"]==0).sum()}')

MURA нийт зураг: 40005
  Хугарал (1): 16403
  Хэвийн  (0): 23602


In [11]:
all_paths = all_mura['path'].tolist()
all_labels = all_mura['label'].tolist()

# Train / Validation / Test huvaalt (80/10/10, stratified)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

print(f'Train: {len(train_paths)} (pos: {sum(train_labels)})')
print(f'Val:   {len(val_paths)} (pos: {sum(val_labels)})')
print(f'Test:  {len(test_paths)} (pos: {sum(test_labels)})')

# FixMatch-d ashiglah: train-aas labeled/unlabeled huvaah
LABELED_RATIO = 0.3  # 30% ni shoshgotoi, 70% ni shoshgoguii
labeled_idx, unlabeled_idx, labeled_labels, _ = train_test_split(
    range(len(train_paths)), train_labels,
    train_size=LABELED_RATIO, random_state=42, stratify=train_labels
)
print(f'\nFixMatch huvaalt:')
print(f'Labeled:   {len(labeled_idx)}')
print(f'Unlabeled: {len(unlabeled_idx)}')

Train: 32007 (pos: 13122)
Val:   4001 (pos: 1641)
Test:  4001 (pos: 1640)

FixMatch huvaalt:
Labeled:   9602
Unlabeled: 22405


In [12]:
# Transforms
# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

# Weak augmentation (FixMatch - pseudo label uusgeh)
weak_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Strong augmentation (FixMatch - surgalt)
strong_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAutocontrast(p=0.3),
    transforms.RandomEqualize(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Validation/Test transform (nemelt uurchlultguii)
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Transforms tayar')

Transforms tayar


In [22]:
class MURADataset(Dataset):
    """MURA dataset class."""
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


class MURAUnlabeledDataset(Dataset):
    """FixMatch-d zoriulsan shoshgoguii dataset.
    Neg zuragaas weak ba strong augmentation-iin 2 huvilbar uusgene."""
    def __init__(self, paths, weak_transform, strong_transform):
        self.paths = paths
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        weak = self.weak_transform(img)
        strong = self.strong_transform(img)
        return weak, strong


# Datasets uusgeh
labeled_paths = [train_paths[i] for i in labeled_idx]
labeled_labs = [train_labels[i] for i in labeled_idx]
unlabeled_paths = [train_paths[i] for i in unlabeled_idx]

labeled_dataset = MURADataset(labeled_paths, labeled_labs, transform=weak_transform)
unlabeled_dataset = MURAUnlabeledDataset(unlabeled_paths, weak_transform, strong_transform)
val_dataset = MURADataset(val_paths, val_labels, transform=eval_transform)
test_dataset = MURADataset(test_paths, test_labels, transform=eval_transform)

# DataLoaders
BATCH_SIZE = 32

labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True, drop_last=True)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Labeled batches: {len(labeled_loader)}')
print(f'Unlabeled batches: {len(unlabeled_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

Labeled batches: 300
Unlabeled batches: 700
Val batches: 126
Test batches: 126


## 3. Model Architecture
EfficientNet-B3 (shared encoder) + CoAtNet-style classification head

In [23]:
class RelativeMultiHeadAttention(nn.Module):
    """CoAtNet-iin self-attention block.
    Orон зайн шинж чанарыг global context-тэй хослуулна."""
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


class CoAtNetBlock(nn.Module):
    """CoAtNet-style block: Conv (local) + Attention (global) hosoluulsan."""
    def __init__(self, dim, num_heads=4, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        # Local: Depthwise Conv
        self.dwconv = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        # Global: Self-Attention
        self.norm1 = nn.LayerNorm(dim)
        self.attn = RelativeMultiHeadAttention(dim, num_heads, dropout)
        # FFN
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Local path
        x = x + self.dwconv(x)
        # Global path (attention)
        x = x + self.attn(self.norm1(x))
        # FFN
        x = x + self.ffn(self.norm2(x))
        return x


class CoAtNetClassificationHead(nn.Module):
    """CoAtNet-style classification head.
    EfficientNet-B3-iin 384-d feature-iig avch, Conv+Attention block-oor
    bolovsruulj, binary classification hiine."""
    def __init__(self, in_features=384, hidden_dim=256, num_heads=4,
                 num_blocks=2, num_classes=1, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_features, hidden_dim)
        self.blocks = nn.ModuleList([
            CoAtNetBlock(hidden_dim, num_heads, dropout=dropout)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        # x: (B, 384)
        x = self.proj(x)           # (B, hidden_dim)
        x = x.unsqueeze(1)         # (B, 1, hidden_dim) - sequence dim nemeh
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        x = x.squeeze(1)           # (B, hidden_dim)
        x = self.head(x)           # (B, 1)
        return x.squeeze(-1)       # (B,)


class FractureClassifier(nn.Module):
    """Buren zagvar: EfficientNet-B3 encoder + CoAtNet classification head."""
    def __init__(self, freeze_encoder=True):
        super().__init__()
        # Shared encoder
        self.encoder = timm.create_model(
            'efficientnet_b3', pretrained=True,
            features_only=False, num_classes=0  # pooled features, no classifier
        )
        encoder_dim = self.encoder.num_features  # 1536 for efficientnet_b3

        # Feature projection (1536 -> 384, shared encoder output dim)
        self.feature_proj = nn.Sequential(
            nn.Linear(encoder_dim, 384),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        # CoAtNet classification head
        self.cls_head = CoAtNetClassificationHead(
            in_features=384, hidden_dim=256,
            num_heads=4, num_blocks=2,
            num_classes=1, dropout=0.3
        )

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
            print('Encoder frozen (jinguudiig holdoosun)')

    def get_features(self, x):
        """384-d features gargah (busad tolgoyd ashiglahad)."""
        with torch.no_grad():
            feat = self.encoder(x)
        return self.feature_proj(feat)

    def forward(self, x):
        feat = self.encoder(x)           # (B, 1536)
        feat = self.feature_proj(feat)    # (B, 384)
        logits = self.cls_head(feat)      # (B,)
        return logits


model = FractureClassifier(freeze_encoder=True).to(device)

# Parametriin too tootsooloh
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Niit parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters: {total_params - trainable_params:,}')

Encoder frozen (jinguudiig holdoosun)
Niit parameters: 12,604,329
Trainable parameters: 1,908,097
Frozen parameters: 10,696,232


## 4. FixMatch Training

In [24]:
# Hyperparameters
NUM_EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
FIXMATCH_THRESHOLD = 0.95   # Pseudo-label confidence threshold
LAMBDA_U = 1.0              # Unlabeled loss weight

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = torch.amp.GradScaler('cuda')  # FP16 mixed precision

print(f'Optimizer: AdamW (lr={LR}, wd={WEIGHT_DECAY})')
print(f'Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})')
print(f'FixMatch threshold: {FIXMATCH_THRESHOLD}')
print(f'Mixed precision: FP16')

Optimizer: AdamW (lr=0.001, wd=0.0001)
Scheduler: CosineAnnealingLR (T_max=30)
FixMatch threshold: 0.95
Mixed precision: FP16


In [25]:
def train_one_epoch_fixmatch(model, labeled_loader, unlabeled_loader,
                              optimizer, scaler, threshold, lambda_u, device):
    """FixMatch surgaltiin neg epoch."""
    model.train()
    total_loss = 0
    total_sup_loss = 0
    total_unsup_loss = 0
    num_batches = 0
    pseudo_used = 0
    pseudo_total = 0

    unlabeled_iter = iter(unlabeled_loader)

    for batch_idx, (images_l, labels_l) in enumerate(labeled_loader):
        images_l = images_l.to(device)
        labels_l = labels_l.float().to(device)

        # Unlabeled batch avah
        try:
            images_uw, images_us = next(unlabeled_iter)
        except StopIteration:
            unlabeled_iter = iter(unlabeled_loader)
            images_uw, images_us = next(unlabeled_iter)

        images_uw = images_uw.to(device)
        images_us = images_us.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            # 1. Supervised loss (labeled data)
            logits_l = model(images_l)
            sup_loss = F.binary_cross_entropy_with_logits(logits_l, labels_l)

            # 2. FixMatch: pseudo-label from weak aug, train on strong aug
            with torch.no_grad():
                logits_uw = model(images_uw)
                probs_uw = torch.sigmoid(logits_uw)
                # Pseudo label: 1 if prob > 0.5, else 0
                pseudo_labels = (probs_uw > 0.5).float()
                # Confidence: max(prob, 1-prob)
                confidence = torch.max(probs_uw, 1 - probs_uw)
                # Mask: confidence > threshold
                mask = (confidence > threshold).float()

            logits_us = model(images_us)
            unsup_loss = (F.binary_cross_entropy_with_logits(
                logits_us, pseudo_labels, reduction='none') * mask).mean()

            # Total loss
            loss = sup_loss + lambda_u * unsup_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_sup_loss += sup_loss.item()
        total_unsup_loss += unsup_loss.item()
        pseudo_used += mask.sum().item()
        pseudo_total += mask.numel()
        num_batches += 1

    avg_loss = total_loss / num_batches
    avg_sup = total_sup_loss / num_batches
    avg_unsup = total_unsup_loss / num_batches
    pseudo_ratio = pseudo_used / max(pseudo_total, 1)

    return avg_loss, avg_sup, avg_unsup, pseudo_ratio


@torch.no_grad()
def evaluate(model, loader, device):
    """Validation/Test unelgee."""
    model.eval()
    all_logits = []
    all_labels = []
    total_loss = 0
    num_batches = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().to(device)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = F.binary_cross_entropy_with_logits(logits, labels)

        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
        total_loss += loss.item()
        num_batches += 1

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels).int()
    all_probs = torch.sigmoid(all_logits)
    all_preds = (all_probs > 0.5).int()

    # Metrics
    auroc = AUROC(task='binary')(all_probs, all_labels).item()
    acc = Accuracy(task='binary')(all_preds, all_labels).item()
    f1 = F1Score(task='binary')(all_preds, all_labels).item()
    prec = Precision(task='binary')(all_preds, all_labels).item()
    rec = Recall(task='binary')(all_preds, all_labels).item()
    avg_loss = total_loss / num_batches

    return {
        'loss': avg_loss, 'auc': auroc, 'accuracy': acc,
        'f1': f1, 'precision': prec, 'recall': rec,
        'probs': all_probs, 'labels': all_labels, 'preds': all_preds
    }

print('Training functions tayar')

Training functions tayar


In [30]:
# FixMatch training loop
best_auc = 0
patience = 5
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'pseudo_ratio': []}

print('='*60)
print('FixMatch surgalt ehelj baina...')
print('='*60)

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, sup_loss, unsup_loss, pseudo_ratio = train_one_epoch_fixmatch(
        model, labeled_loader, unlabeled_loader,
        optimizer, scaler, FIXMATCH_THRESHOLD, LAMBDA_U, device
    )

    # Validate
    val_metrics = evaluate(model, val_loader, device)

    # Scheduler step
    scheduler.step()

    # History
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_auc'].append(val_metrics['auc'])
    history['pseudo_ratio'].append(pseudo_ratio)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} (sup: {sup_loss:.4f}, unsup: {unsup_loss:.4f}) | "
          f"Val Loss: {val_metrics['loss']:.4f} | "
          f"Val AUC: {val_metrics['auc']:.4f} | "
          f"Val Acc: {val_metrics['accuracy']:.4f} | "
          f"Pseudo used: {pseudo_ratio:.1%}")

    # Best model save
    if val_metrics['auc'] > best_auc:
        best_auc = val_metrics['auc']
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_auc': best_auc,
        }, os.path.join(SAVE_DIR, 'best_model_fixmatch.pth'))
        print(f'  -> Shine hamgiin sain zagvar hadgalav (AUC: {best_auc:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'\nSurgalt duuslaa. Best Val AUC: {best_auc:.4f}')

FixMatch surgalt ehelj baina...


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c5ca83d3600>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c5ca83d3600>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_812/1703866808.py", line 31, in __getitem__
    img = Image.open(self.paths[idx]).convert('RGB')
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/MURA/MURA-v1.1/train/XR_WRIST/patient07840/study2_negative/._image1.png'


## 5. Evaluation & Results

In [ ]:
# Hamgiin sain zagvariig achaalah
checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model_fixmatch.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Best model achaallaa (Epoch {checkpoint['epoch']+1}, AUC: {checkpoint['best_auc']:.4f})")

# Test set deer uneleh
test_metrics = evaluate(model, test_loader, device)

print('\n' + '='*50)
print('TEST SET UR DUN')
print('='*50)
print(f"AUC:         {test_metrics['auc']:.4f}")
print(f"Accuracy:    {test_metrics['accuracy']:.4f}")
print(f"F1-Score:    {test_metrics['f1']:.4f}")
print(f"Precision:   {test_metrics['precision']:.4f}")
print(f"Recall/Sens: {test_metrics['recall']:.4f}")
print(f"Specificity: {1 - (1 - test_metrics['precision']):.4f}")

In [ ]:
# Surgaltiin history graph
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC
axes[1].plot(history['val_auc'], label='Val AUC', color='green')
axes[1].axhline(y=0.85, color='red', linestyle='--', label='Target (0.85)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('Validation AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Pseudo label usage
axes[2].plot(history['pseudo_ratio'], label='Pseudo Label Ratio', color='orange')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Ratio')
axes[2].set_title('FixMatch Pseudo Label Usage')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Graph hadgalagdlaa')

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, classification_report
import seaborn as sns

# Confusion Matrix
cm = confusion_matrix(test_metrics['labels'].numpy(), test_metrics['preds'].numpy())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Abnormal'],
            yticklabels=['Normal', 'Abnormal'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(test_metrics['labels'].numpy(), test_metrics['probs'].numpy())
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['auc']:.4f}", color='blue')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'evaluation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

# Classification report
print('\nClassification Report:')
print(classification_report(
    test_metrics['labels'].numpy(),
    test_metrics['preds'].numpy(),
    target_names=['Normal', 'Abnormal']
))

## 6. Baseline Comparison (Supervised Only)

In [ ]:
# FixMatch-guigeer, zovhon labeled data deer surgasan baseline
# Haritsuulahiin tuld supervised-only zagvar surgana

print('Baseline (supervised-only) zagvar surgaj baina...')

baseline_model = FractureClassifier(freeze_encoder=True).to(device)
baseline_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, baseline_model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
baseline_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(baseline_optimizer, T_max=NUM_EPOCHS)
baseline_scaler = torch.amp.GradScaler('cuda')

best_baseline_auc = 0
baseline_patience = 0

for epoch in range(NUM_EPOCHS):
    baseline_model.train()
    epoch_loss = 0
    n_batches = 0

    for images, labels in labeled_loader:
        images = images.to(device)
        labels = labels.float().to(device)

        baseline_optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = baseline_model(images)
            loss = F.binary_cross_entropy_with_logits(logits, labels)

        baseline_scaler.scale(loss).backward()
        baseline_scaler.step(baseline_optimizer)
        baseline_scaler.update()

        epoch_loss += loss.item()
        n_batches += 1

    baseline_scheduler.step()
    val_m = evaluate(baseline_model, val_loader, device)

    if val_m['auc'] > best_baseline_auc:
        best_baseline_auc = val_m['auc']
        baseline_patience = 0
        torch.save(baseline_model.state_dict(),
                   os.path.join(SAVE_DIR, 'best_model_baseline.pth'))
    else:
        baseline_patience += 1
        if baseline_patience >= 5:
            print(f'Baseline early stopping at epoch {epoch+1}')
            break

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}] Loss: {epoch_loss/n_batches:.4f} | "
              f"Val AUC: {val_m['auc']:.4f} | Val Acc: {val_m['accuracy']:.4f}")

# Baseline test
baseline_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'best_model_baseline.pth')))
baseline_test = evaluate(baseline_model, test_loader, device)

print('\n' + '='*50)
print('HARITSUULALT: FixMatch vs Supervised-Only')
print('='*50)
print(f'{"Hemjuur":<15} {"Supervised":>12} {"FixMatch":>12} {"Yalgaa":>10}')
print('-'*50)
for metric in ['auc', 'accuracy', 'f1', 'precision', 'recall']:
    b = baseline_test[metric]
    f = test_metrics[metric]
    diff = f - b
    print(f'{metric:<15} {b:>12.4f} {f:>12.4f} {diff:>+10.4f}')

## 7. Grad-CAM Visualization

In [ ]:
!pip install -q pytorch-grad-cam

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

# Grad-CAM target layer (encoder-iin suuliin conv layer)
target_layers = [model.encoder.conv_head]

cam = GradCAM(model=model, target_layers=target_layers)

def visualize_gradcam(model, dataset, indices, save_path=None):
    """Songogdson zurguudiin Grad-CAM duulaaany zuraglalyg haruulna."""
    fig, axes = plt.subplots(len(indices), 3, figsize=(12, 4*len(indices)))
    if len(indices) == 1:
        axes = axes.reshape(1, -1)

    for i, idx in enumerate(indices):
        img_tensor, label = dataset[idx]

        # Original image (denormalize)
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)

        # Prediction
        input_tensor = img_tensor.unsqueeze(0).to(device)
        with torch.no_grad():
            logit = model(input_tensor)
            prob = torch.sigmoid(logit).item()
            pred = 1 if prob > 0.5 else 0

        # Grad-CAM
        targets = [BinaryClassifierOutputTarget(1)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        cam_image = show_cam_on_image(img_np.astype(np.float32), grayscale_cam, use_rgb=True)

        # Plot
        true_str = 'Abnormal' if label == 1 else 'Normal'
        pred_str = 'Abnormal' if pred == 1 else 'Normal'
        color = 'green' if pred == label else 'red'

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f'Original (True: {true_str})')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(cam_image)
        axes[i, 1].set_title(f'Grad-CAM')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(grayscale_cam, cmap='jet')
        axes[i, 2].set_title(f'Pred: {pred_str} ({prob:.2f})', color=color)
        axes[i, 2].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# 6 zurag songoj haruulah (3 abnormal, 3 normal)
abnormal_indices = [i for i, l in enumerate(test_labels) if l == 1][:3]
normal_indices = [i for i, l in enumerate(test_labels) if l == 0][:3]
sample_indices = abnormal_indices + normal_indices

visualize_gradcam(model, test_dataset, sample_indices,
                  save_path=os.path.join(SAVE_DIR, 'gradcam_results.png'))
print('Grad-CAM zurag hadgalagdlaa')

## 8. Save Features for Other Heads

In [ ]:
# 384-d features hadgalah (segmentation, severity, healing head-uudad ashiglahad)
# Encoder-iin garltiig feature_proj-oor damjuulj 384-d bolgono

@torch.no_grad()
def extract_features(model, loader, device):
    """Encoder + feature_proj-oor 384-d features gargaj avna."""
    model.eval()
    all_features = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        feat = model.encoder(images)         # (B, 1536)
        feat = model.feature_proj(feat)       # (B, 384)
        all_features.append(feat.cpu().numpy())
        all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)

print('MURA features yalgaj baina...')
# Full dataset loader (augmentation-guigeer)
full_dataset = MURADataset(all_paths, all_labels, transform=eval_transform)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)

mura_features, mura_labels = extract_features(model, full_loader, device)

np.save(os.path.join(SAVE_DIR, 'mura_features.npy'), mura_features)
np.save(os.path.join(SAVE_DIR, 'mura_labels.npy'), mura_labels)

print(f'Features hadgalagdlaa: {mura_features.shape}')
print(f'Labels hadgalagdlaa: {mura_labels.shape}')
print(f'Feature dim: {mura_features.shape[1]}')

In [ ]:
# Etsiin negdsen ur dun
print('\n' + '='*60)
print('NEGDSEN UR DUN')
print('='*60)
print(f'Zagvar: EfficientNet-B3 + CoAtNet Head + FixMatch')
print(f'Dataset: MURA (~{len(all_paths):,} zurag)')
print(f'Labeled ratio: {LABELED_RATIO:.0%}')
print(f'FixMatch threshold: {FIXMATCH_THRESHOLD}')
print(f'\nTest Results:')
print(f"  AUC:         {test_metrics['auc']:.4f}")
print(f"  Accuracy:    {test_metrics['accuracy']:.4f}")
print(f"  F1-Score:    {test_metrics['f1']:.4f}")
print(f"  Precision:   {test_metrics['precision']:.4f}")
print(f"  Sensitivity: {test_metrics['recall']:.4f}")
print(f'\nZorilttoi haritsuulbal:')
print(f"  AUC >= 0.85: {'HURSEN' if test_metrics['auc'] >= 0.85 else 'HUREEGUII'}")
print(f'\nHadgalsan file-uud:')
for f in os.listdir(SAVE_DIR):
    fpath = os.path.join(SAVE_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f'  {f} ({size_mb:.1f} MB)')